# Baidu ULTR Parser Checks

This notebook parses Baidu ULTR using `feature_based_propensities_for_ULTR.data.parsers.baidu_ultr` and prints deterministic inspection tables (first 10 rows each time) so you can quickly verify everything is working.

In [7]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
from IPython.display import display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "feature_based_propensities_for_ULTR").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing 'feature_based_propensities_for_ULTR'.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from feature_based_propensities_for_ULTR.data.parsers.baidu_ultr import (  # noqa: E402
    DEFAULT_MAX_LEN,
    discover_baidu_files,
    parse_and_save_baidu_ultr,
)
from feature_based_propensities_for_ULTR.data.parsers.contracts import (  # noqa: E402
    validate_prebuilt_click_dataset,
)

HEAD_N = 20
pd.set_option("display.max_rows", HEAD_N)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/stanf/Documents/Uni/MSc Thesis/doubly-robust-optim-with-Two-tower-Models


In [2]:
DEFAULT_PARTS_DIR = Path(
    os.environ.get("BAIDU_PARTS_DIR", "/Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts")
).expanduser()
DEFAULT_OUTPUT_DIR = Path(
    os.environ.get("BAIDU_OUTPUT_DIR", str(PROJECT_ROOT / "data" / "baidu_ultr" / "processed"))
).expanduser()
MAX_LEN = int(os.environ.get("BAIDU_MAX_LEN", str(DEFAULT_MAX_LEN)))
OVERWRITE = str(os.environ.get("BAIDU_OVERWRITE", "1")).strip().lower() in {"1", "true", "yes", "y"}

PARTS_DIR = DEFAULT_PARTS_DIR
OUTPUT_DIR = DEFAULT_OUTPUT_DIR

print("Configuration")
print(f"  PARTS_DIR : {PARTS_DIR}")
print(f"  OUTPUT_DIR: {OUTPUT_DIR}")
print(f"  MAX_LEN   : {MAX_LEN}")
print(f"  OVERWRITE : {OVERWRITE}")

if not PARTS_DIR.exists():
    raise FileNotFoundError(
        f"PARTS_DIR does not exist: {PARTS_DIR}. Set BAIDU_PARTS_DIR or update PARTS_DIR in this cell."
    )


Configuration
  PARTS_DIR : /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts
  OUTPUT_DIR: /Users/stanf/Documents/Uni/MSc Thesis/doubly-robust-optim-with-Two-tower-Models/data/baidu_ultr/processed
  MAX_LEN   : 20
  OVERWRITE : True


In [3]:
def stable_head(df: pd.DataFrame, sort_cols: list[str], n: int = HEAD_N) -> pd.DataFrame:
    usable = [c for c in sort_cols if c in df.columns]
    if usable:
        return df.sort_values(usable, kind="mergesort").head(n).reset_index(drop=True)
    return df.head(n).reset_index(drop=True)


def feather_columns(path: Path) -> list[str]:
    try:
        with pa.memory_map(str(path), "r") as source:
            reader = pa.ipc.RecordBatchFileReader(source)
            return list(reader.schema.names)
    except Exception:
        return list(feather.read_table(str(path)).column_names)


def raw_table_preview(path: Path, n: int = HEAD_N) -> pd.DataFrame:
    preferred = [
        "query_no",
        "query_md5",
        "text_md5",
        "position",
        "click",
        "label",
        "bm25",
        "query_length",
    ]
    columns = feather_columns(path)
    selected = [c for c in preferred if c in columns]
    if not selected:
        selected = columns[: min(10, len(columns))]
    table = feather.read_table(str(path), columns=selected)
    return table.slice(0, n).to_pandas().reset_index(drop=True)


def split_preview(npz_path: Path, n: int = HEAD_N) -> tuple[pd.DataFrame, pd.DataFrame]:
    with np.load(npz_path, allow_pickle=False) as data:
        sessions = np.asarray(data["sessions"])
        query = np.asarray(data["query"])
        positions = np.asarray(data["positions"])
        clicks = np.asarray(data["clicks"])
        labels = np.asarray(data["labels"])
        mask = np.asarray(data["mask"], dtype=bool)
        query_doc_ids = np.asarray(data["query_doc_ids"])
        n_docs = np.asarray(data["n"])
        qdoc = np.asarray(data["query_doc_features"])
        lp = np.asarray(data["lp_query_doc_features"])

    max_len_local = positions.shape[1]

    def build_row(session_idx: int, rank_idx: int) -> dict[str, object]:
        return {
            "session": int(sessions[session_idx]),
            "query": int(query[session_idx]),
            "rank": int(positions[session_idx, rank_idx]),
            "mask": bool(mask[session_idx, rank_idx]),
            "click": int(clicks[session_idx, rank_idx]),
            "label": float(labels[session_idx, rank_idx]),
            "query_doc_id": int(query_doc_ids[session_idx, rank_idx]),
            "n_docs_in_session": int(n_docs[session_idx]),
            "bm25": float(qdoc[session_idx, rank_idx, 0]),
            "lp_position_feature": float(lp[session_idx, rank_idx, 0]),
        }

    all_rows: list[dict[str, object]] = []
    for session_idx in range(len(sessions)):
        for rank_idx in range(max_len_local):
            all_rows.append(build_row(session_idx, rank_idx))
            if len(all_rows) == n:
                break
        if len(all_rows) == n:
            break

    valid_rows: list[dict[str, object]] = []
    for session_idx in range(len(sessions)):
        for rank_idx in range(max_len_local):
            if not mask[session_idx, rank_idx]:
                continue
            valid_rows.append(build_row(session_idx, rank_idx))
            if len(valid_rows) == n:
                break
        if len(valid_rows) == n:
            break

    return pd.DataFrame(all_rows), pd.DataFrame(valid_rows)


In [8]:
train_paths, val_paths, labels_path, schema_columns = discover_baidu_files(PARTS_DIR)

display(
    pd.DataFrame(
        {
            "split": ["train", "val", "test(labels)"],
            "num_files": [len(train_paths), len(val_paths), 1],
            "first_file": [
                str(train_paths[0]) if train_paths else "-",
                str(val_paths[0]) if val_paths else "-",
                str(labels_path),
            ],
        }
    )
)

print(f"Schema columns (excluding query_document_embedding): {len(schema_columns)}")
display(stable_head(pd.DataFrame({"column": schema_columns}), ["column"], n=HEAD_N))

print(f"Train raw preview ({HEAD_N} rows from first train file)")
display(raw_table_preview(train_paths[0], n=HEAD_N))

print(f"Val raw preview ({HEAD_N} rows from first val file)")
display(raw_table_preview(val_paths[0], n=HEAD_N))

print(f"Validation labels raw preview ({HEAD_N} rows)")
display(raw_table_preview(labels_path, n=HEAD_N))


,split,num_files,first_file
0,train,30,/Users/stanf/Documents/Uni/Project_AI/baidu_ul...
1,val,10,/Users/stanf/Documents/Uni/Project_AI/baidu_ul...
2,test(labels),1,/Users/stanf/Documents/Uni/Project_AI/baidu_ul...


Schema columns (excluding query_document_embedding): 28


,column
0,__index_level_0__
1,abstract
2,abstract_length
3,bm25
4,bm25_abstract
5,bm25_title
6,click
7,displayed_time
8,document_length
9,idf


Train raw preview (20 rows from first train file)


,query_no,query_md5,text_md5,position,click,bm25,query_length
0,0,604220ba286ee97bd11ef9047a554cab,1efda338f873b8484f0b748382bb1bb4,1,0,37.583162,12
1,0,604220ba286ee97bd11ef9047a554cab,f54bd833573c503a91997c141b4317a2,2,0,36.205723,12
2,0,604220ba286ee97bd11ef9047a554cab,57cdbb2741d487ef67a525193b8cbb41,3,0,56.078710,12
3,0,604220ba286ee97bd11ef9047a554cab,3ea809b3cfe85660d16e5de7925572a9,4,0,70.858204,12
4,0,604220ba286ee97bd11ef9047a554cab,2d9d9634d0d6d979744f9cbe289aecd0,6,0,52.380690,12
5,0,604220ba286ee97bd11ef9047a554cab,28bb49c02b18d09ea72690478ca43b9e,7,0,47.444211,12
6,0,604220ba286ee97bd11ef9047a554cab,dbfd2e0c40d8f3c01af8ae116af68b71,8,0,51.777689,12
7,0,604220ba286ee97bd11ef9047a554cab,970afc0a8b8ec49b9d359e8993f31098,9,0,41.325139,12
8,1,00ddb3440ce04d3fad21494083591b6a,eb00f6ac0a2b3c76744aaa6b8174a81e,1,1,22.760116,5
9,1,00ddb3440ce04d3fad21494083591b6a,1d29d6fcf2f3e3d7cfe3134f7498ce52,2,1,25.614769,5


Val raw preview (20 rows from first val file)


,query_no,query_md5,text_md5,position,click,bm25,query_length
0,0,e63b18d9947960147a9cf9acd47d8d10,3000d7e94b1567c14c1856eb95c7ef5c,1,0,29.704134,7
1,0,e63b18d9947960147a9cf9acd47d8d10,b6cbbc290a16d0cfd48b47d9874263db,2,1,26.743759,7
2,0,e63b18d9947960147a9cf9acd47d8d10,dd716130f51a9bc95483ad34393a9f2a,3,1,16.636720,7
3,0,e63b18d9947960147a9cf9acd47d8d10,6fb0c3169c213087e87888e00517abdc,4,0,28.786646,7
4,0,e63b18d9947960147a9cf9acd47d8d10,6eeccab8210e4f15b7bf252d43c6326c,5,0,28.616010,7
5,0,e63b18d9947960147a9cf9acd47d8d10,3ee5a7800a896c5bfd6305e4985e1bf6,6,0,28.232823,7
6,0,e63b18d9947960147a9cf9acd47d8d10,bcae33003929284a3c27591e89b406dd,7,0,25.417476,7
7,0,e63b18d9947960147a9cf9acd47d8d10,3374859f47d738e6d7e558bdfd4a93b0,8,0,28.616010,7
8,0,e63b18d9947960147a9cf9acd47d8d10,7ffdaeca876228130b4cb86a29756e44,9,0,30.783193,7
9,1,c5eac96413778f5ef75fd736246ff39d,0b5f04f5f38cd0d0baf8602e28a6ad80,1,1,43.301189,9


Validation labels raw preview (20 rows)


,query_no,query_md5,text_md5,label,bm25,query_length
0,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,e9d6cf386e9944c3dad128661413ac8f,3,29.989558,11
1,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,26235faf6c49bbf72dbb70321a08af1b,0,25.080365,11
2,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,96f506d2ed181af864eeca28d9e739e2,2,28.417633,11
3,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,604a275f0486e60b41771e6d8ed73939,2,23.151721,11
4,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,8adf2b39226c2de8fd6e5d02113c15f8,0,25.179724,11
5,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,23a0ec0b9d6b54ff276fe0c244bca3a6,2,27.747499,11
6,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,c16024266ed879d995e960ec23a2c8cf,3,32.864661,11
7,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,90f549344f9b80971f793dbee6820a42,2,32.165457,11
8,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,8d5a99b0ff5fa1fd99311c919ec84b6c,1,28.212342,11
9,0,ad7b0646d3a2dfcb23c8fdf7fca19fad,ff3609a2d9332cc7e10eb88391fe4fe7,2,27.277775,11


In [5]:
manifest = parse_and_save_baidu_ultr(
    parts_dir=PARTS_DIR,
    output_dir=OUTPUT_DIR,
    max_len=MAX_LEN,
    overwrite=OVERWRITE,
    pad_to_max_len=True,
)

print("Manifest summary")
print(
    json.dumps(
        {
            "parts_dir": manifest["parts_dir"],
            "output_dir": manifest["output_dir"],
            "max_len": manifest["max_len"],
            "padding_strategy": manifest["padding_strategy"],
            "created_at_utc": manifest["created_at_utc"],
        },
        indent=2,
    )
)
display(pd.DataFrame(manifest["stats"]).T)


Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-0.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-1.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-2.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-3.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-4.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-5.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-6.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-7.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-8.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/baidu_ultr_dataset/parts/part-1_split-9.feather
Loading: /Users/stanf/Documents/Uni/Project_AI/bai

,name,files,rows_total,rows_kept,rows_dropped_due_to_max_len,rows_dropped_invalid_position,rows_dropped_duplicate_position,sessions,max_docs_per_session_observed
train,train,[/Users/stanf/Documents/Uni/Project_AI/baidu_u...,14526276,14526272,4,0,0,1779017,20
val,val,[/Users/stanf/Documents/Uni/Project_AI/baidu_u...,4848878,4848877,1,0,0,593930,20
test,test,[/Users/stanf/Documents/Uni/Project_AI/baidu_u...,397522,103330,294192,0,0,5182,20


In [9]:
validation_summary = validate_prebuilt_click_dataset(OUTPUT_DIR)
print("Validation summary")
display(pd.DataFrame(validation_summary).T)

for split in ("train", "val", "test"):
    npz_path = OUTPUT_DIR / f"{split}_click_dataset.npz"
    all_rows, valid_rows = split_preview(npz_path, n=HEAD_N)

    print(f"\n{split}: first {HEAD_N} rows (natural session/rank order; includes padded rows)")
    display(all_rows)

    print(f"{split}: first {HEAD_N} valid rows (mask=True)")
    display(valid_rows)


Validation summary


,sessions,rows_kept,max_docs_per_session,total_clicks
train,1779017,14526272,20,1224140
val,593930,4848877,20,409086
test,5182,103330,20,0



train: first 20 rows (natural session/rank order; includes padded rows)


,session,query,rank,mask,click,label,query_doc_id,n_docs_in_session,bm25,lp_position_feature
0,0,4142184726982758121,1,True,0,0.0,0,8,37.583164,1.0
1,0,4142184726982758121,2,True,0,0.0,1,8,36.205723,2.0
2,0,4142184726982758121,3,True,0,0.0,2,8,56.078709,3.0
3,0,4142184726982758121,4,True,0,0.0,3,8,70.858200,4.0
4,0,4142184726982758121,5,False,0,0.0,-1,8,0.000000,0.0
5,0,4142184726982758121,6,True,0,0.0,4,8,52.380692,6.0
6,0,4142184726982758121,7,True,0,0.0,5,8,47.444210,7.0
7,0,4142184726982758121,8,True,0,0.0,6,8,51.777691,8.0
8,0,4142184726982758121,9,True,0,0.0,7,8,41.325138,9.0
9,0,4142184726982758121,10,False,0,0.0,-1,8,0.000000,0.0


train: first 20 valid rows (mask=True)


...

,session,query,rank,mask,click,label,query_doc_id,n_docs_in_session,bm25,lp_position_feature
0,0,8041100333397439461,1,True,0,0.0,0,9,29.704134,1.0
1,0,8041100333397439461,2,True,1,1.0,1,9,26.743759,2.0
2,0,8041100333397439461,3,True,1,1.0,2,9,16.636721,3.0
3,0,8041100333397439461,4,True,0,0.0,3,9,28.786646,4.0
4,0,8041100333397439461,5,True,0,0.0,4,9,28.616011,5.0
5,0,8041100333397439461,6,True,0,0.0,5,9,28.232822,6.0
6,0,8041100333397439461,7,True,0,0.0,6,9,25.417477,7.0
7,0,8041100333397439461,8,True,0,0.0,7,9,28.616011,8.0
8,0,8041100333397439461,9,True,0,0.0,8,9,30.783194,9.0
9,1,4963175667083940384,1,True,1,1.0,9,9,43.301189,1.0



test: first 20 rows (natural session/rank order; includes padded rows)


,session,query,rank,mask,click,label,query_doc_id,n_docs_in_session,bm25,lp_position_feature
0,0,7915789230678463750,1,True,0,3.0,0,20,29.989557,1.0
1,0,7915789230678463750,2,True,0,0.0,1,20,25.080364,2.0
2,0,7915789230678463750,3,True,0,2.0,2,20,28.417633,3.0
3,0,7915789230678463750,4,True,0,2.0,3,20,23.151722,4.0
4,0,7915789230678463750,5,True,0,0.0,4,20,25.179724,5.0
5,0,7915789230678463750,6,True,0,2.0,5,20,27.747499,6.0
6,0,7915789230678463750,7,True,0,3.0,6,20,32.864662,7.0
7,0,7915789230678463750,8,True,0,2.0,7,20,32.165459,8.0
8,0,7915789230678463750,9,True,0,1.0,8,20,28.212343,9.0
9,0,7915789230678463750,10,True,0,2.0,9,20,27.277775,10.0


test: first 20 valid rows (mask=True)


,session,query,rank,mask,click,label,query_doc_id,n_docs_in_session,bm25,lp_position_feature
0,0,7915789230678463750,1,True,0,3.0,0,20,29.989557,1.0
1,0,7915789230678463750,2,True,0,0.0,1,20,25.080364,2.0
2,0,7915789230678463750,3,True,0,2.0,2,20,28.417633,3.0
3,0,7915789230678463750,4,True,0,2.0,3,20,23.151722,4.0
4,0,7915789230678463750,5,True,0,0.0,4,20,25.179724,5.0
5,0,7915789230678463750,6,True,0,2.0,5,20,27.747499,6.0
6,0,7915789230678463750,7,True,0,3.0,6,20,32.864662,7.0
7,0,7915789230678463750,8,True,0,2.0,7,20,32.165459,8.0
8,0,7915789230678463750,9,True,0,1.0,8,20,28.212343,9.0
9,0,7915789230678463750,10,True,0,2.0,9,20,27.277775,10.0
